In [ ]:
# %% Deep learning - Section 19.182
#    Code challenge 31: how low can you go?
#
#    1) Start from code from video 19.184 (emnist dataset)
#    2) Improve the performance shown in the video
#    3) Consider number of layers, kernel sizes, number of units, batch norm,
#       dropout, L1/L2 regularisation, data augmentation, etc.
#    4) Ideally get an error rate lower than 8%, even better if less than 5%

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [ ]:
# %% Tried solution

# Add dropout (less for convolutional layers than fully-connected), implement
# some data augmentation, modify model architecture in two Conv-Conv-Pool blocks
# with more kernels


In [2]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [54]:
# %% Data

# Load data
# See also : https://www.nist.gov/itl/products-and-services/emnist-dataset
data = torchvision.datasets.EMNIST(root='emnist',split='letters',download=True)


In [57]:
# %% Preprocess data

# Transform into 4D tensor for convolution layers, transform from int8 to float
images = data.data.view([124800,1,28,28]).float()

# Remove N/A (first class category; relabel to start at 0)
letter_categories = data.classes[1:]
labels = copy.deepcopy(data.targets)-1

# Normalise
images /= torch.max(images)


In [61]:
class CustomDataset(Dataset):

    def __init__(self,tensors,transform=None):

        # Check size match
        assert tensors[0].size(0) == tensors[1].size(0), "Size mismatch"

        # Store tensors
        self.tensors   = tensors
        self.transform = transform

    def __getitem__(self,index):

        x = self.tensors[0][index]

        if self.transform is not None:
            x = self.transform(x)

        y = self.tensors[1][index]

        return x,y

    def __len__(self):

        return self.tensors[0].size(0)


In [62]:
# %% Training and test transforms

# Random rotations from uniform distribution, and translation plus shear
train_transforms = T.Compose([ T.ToPILImage(),
                               T.RandomRotation((-15,15)),
                               T.RandomAffine(degrees=0,translate=(0.1,0.1),shear=10),
                               T.ToTensor() ])

test_transforms = T.Compose([ T.ToPILImage(),
                              T.ToTensor() ])


In [63]:
# %% Create train and test datasets

# Split data with scikitlearn (10% test data)
train_data,test_data,train_labels,test_labels = train_test_split(images,labels,test_size=0.1)

# Convert to PyTorch datasets
train_data = CustomDataset((train_data,train_labels),transform=train_transforms)
test_data  = CustomDataset((test_data,test_labels),transform=test_transforms)

# Convert into DataLoader objects
batch_size   = 32
train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True,drop_last=True)
test_loader  = DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])


In [ ]:
# Plotting

X,y = next(iter(train_loader))

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(3,7,figsize=(phi*6,6))

for i,ax in enumerate(axs.flatten()):

    whichpic = np.random.randint(X.shape[0])

    I = X[whichpic,0,:,:].numpy()
    letter = letter_categories[y[whichpic]]

    ax.imshow(I.T,cmap='gray')
    ax.set_title('The letter "%s"'%letter)
    ax.set_xticks([])
    ax.set_yticks([])

plt.savefig('figure126_code_challenge_31.png')
plt.show()
files.download('figure126_code_challenge_31.png')


In [71]:
# %% Function to generate the model

def gen_model():

    class emnist_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Use blocks of Conv-Conv-Pool (VGG style)
            # Convolution block 1
            self.conv1_1 = nn.Conv2d(1,32,kernel_size=3,padding=1)
            self.conv1_2 = nn.Conv2d(32,32,kernel_size=3,padding=1)

            # Batch normalisation and dropout for convolution
            self.bnorm1 = nn.BatchNorm2d(32)
            self.drop1  = nn.Dropout2d(p=0.15)

            # Convolution block 2
            self.conv2_1 = nn.Conv2d(32,64,kernel_size=3,padding=1)
            self.conv2_2 = nn.Conv2d(64,64,kernel_size=3,padding=1)

            # Batch normalisation and dropout for convolution
            self.bnorm2 = nn.BatchNorm2d(64)
            self.drop2  = nn.Dropout2d(p=0.15)

            # Fully connected layer with batch normalisation
            self.fc1       = nn.Linear(7*7*64,50)
            self.bnorm_fc1 = nn.BatchNorm1d(50)
            self.drop_fc1  = nn.Dropout(p=0.5)

            # Output layer
            self.output = nn.Linear(50,26)

        def forward(self,x):

            # MaxPool and ReLu on convolution block 1 (pool after passing to
            # activation function)
            x = F.leaky_relu(self.conv1_1(x))
            x = F.leaky_relu(self.conv1_2(x))
            x = F.max_pool2d(x,2)
            x = self.bnorm1(x)
            x = self.drop1(x)

            # MaxPool and ReLu on convolution block 2
            x = F.leaky_relu(self.conv2_1(x))
            x = F.leaky_relu(self.conv2_2(x))
            x = F.max_pool2d(x, 2)
            x = self.bnorm2(x)
            x = self.drop2(x)

            # Vectorise for linear layer
            n_units = x.shape.numel() / x.shape[0]
            x       = x.view(-1,int(n_units))

            # Linear and output layers
            x = F.leaky_relu(self.fc1(x))
            x = self.bnorm_fc1(x)

            x = self.output(x)

            return x

    # Create model instance
    CNN = emnist_CNN()

    # Loss function
    loss_fun = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

CNN,loss_fun,optimizer = gen_model()

X,y  = next(iter(train_loader))
yHat = CNN(X)

# Check sizes of output and target variable
print()
print(yHat.shape), print()
print(y.shape), print()

# Check loss
loss = loss_fun(yHat,y)
print(loss)


In [73]:
# %% Function to train the model

def train_model():

    # Parameters, model instance, inizialise vars
    num_epochs = 10
    CNN,loss_fun,optimizer = gen_model()

    # Ship model to GPU
    CNN.to(device)

    train_losses = []
    test_losses  = []
    train_acc    = []
    test_acc     = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # Loop over training batches
        batch_acc  = []
        batch_loss = []

        for X,y in train_loader:

            # Ship data to GPU
            X = X.to(device)
            y = y.to(device)

            # Forward propagation and loss
            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and error rate (1-accuracy) from this batch
            yHat = yHat.cpu()
            y    = y.cpu()

            batch_loss.append(loss.item())
            batch_acc.append( torch.mean( (torch.argmax(yHat,axis=1) != y).float() ).item() )

        train_losses.append( np.mean(batch_loss) )
        train_acc.append( 100*np.mean(batch_acc) )

        # Test accuracy
        CNN.eval()

        with torch.no_grad():
            X,y = next(iter(test_loader))

            # Ship to GPU (y for loss calculation)
            X = X.to(device)
            y = y.to(device)

            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            yHat = yHat.cpu()
            y    = y.cpu()

            test_acc.append( 100*torch.mean( (torch.argmax(yHat,axis=1) != y).float() ).item() )
            test_losses.append(loss.item())

        CNN.train()

    return train_acc,test_acc,train_losses,test_losses,CNN


In [74]:
# %% Fit model

# Takes ~7 mins on GPU
train_err,test_err,train_losses,test_losses,CNN = train_model()


In [ ]:
# Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_losses,'s-',label='Train')
ax[0].plot(test_losses,'o-',label='Test')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')

ax[1].plot(train_err,'s-',label='Train')
ax[1].plot(test_err,'o-',label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Error rates (%)')
ax[1].set_title(f'Final model test error rate: {test_err[-1]:.2f}%')
ax[1].legend()

plt.savefig('figure127_code_challenge_31.png')
plt.show()
files.download('figure127_code_challenge_31.png')


In [ ]:
# %% Plotting

# get data from test dataloader
X,y  = next(iter(test_loader))
X    = X.to(device)
y    = y.to(device)
yHat = CNN(X)

# Pick random examples
rand_id = np.random.choice(len(y),size=21,replace=False)

# Plot
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(3,7,figsize=(1.5*phi*6,6))

for i,ax in enumerate(axs.flatten()):

    # Extract image and its target letter; .cpu() to transfer back from GPU
    I = np.squeeze( X[rand_id[i],0,:,:] ).cpu()
    true_letter = letter_categories[ y[rand_id[i]] ]
    pred_letter = letter_categories[ torch.argmax(yHat[rand_id[i],:]) ]

    # Colour-code wrong classification (with ternary operator)
    col = 'gray' if true_letter==pred_letter else 'hot'

    ax.imshow(I.T,cmap=col)
    ax.set_title('True %s, predicted %s' %(true_letter,pred_letter),fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.savefig('figure128_code_challenge_31.png')
plt.show()
files.download('figure128_code_challenge_31.png')


In [ ]:
# %% So many classes! Compute the confusion matrix

# Confusion matrix
C = skm.confusion_matrix(y.cpu(),torch.argmax(yHat.cpu(),axis=1),normalize='true')

# Plot
phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(7*phi,7))
plt.imshow(C,'Blues',vmax=.05)

# make the plot look nicer
plt.xticks(range(26),labels=letter_categories)
plt.yticks(range(26),labels=letter_categories)
plt.title('Confusion matrix\nTest data')
plt.xlabel('True letter')
plt.xlabel('Predicted letter')
plt.ylabel('True number')
plt.colorbar()

plt.savefig('figure129_code_challenge_31.png')
plt.show()
files.download('figure129_code_challenge_31.png')
